# Gemma 3 12B-IT — проверка на RunPod A40

`google/gemma-3-12b-it` — из исходного плана как запасной вариант (там рассматривалась Gemma 3 4B).
На A40 можно позволить 12B вместо 4B — честнее показывает потолок семейства.

**Казахский нигде явно не заявлен.** Карточка модели говорит про "140+ языков" общей фразой, без списка —
это ровно то предупреждение, что было в исходном плане ("казахский не подтверждён официально").
Эта проверка — не поиск ещё одного кандидата на казахский, а честная проверка: работает ли общее заявление
на практике, или Gemma на kk заметно слабее специализированных Qolda/Sherkala/KazLLM.

Зрение есть (Gemma 3 мультимодальна), тоже стоит проверить на том же тестовом документе.

**Доступ:** gated (manual license acceptance, не auto как у Sherkala/KazLLM) — на странице модели нужно принять
лицензию Google, обычно тоже выдаётся сразу, но форма/процесс немного другой.

In [ ]:
# --- 0. Проверка GPU ---
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('vram total GB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
    print('vram free GB:', round((torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 2**30, 1))
else:
    raise SystemExit('GPU недоступна в этом контейнере.')

In [ ]:
# --- HF-токен ---
# Тот же токен, что для Sherkala/KazLLM, при условии что лицензия Gemma принята отдельно
# на huggingface.co/google/gemma-3-12b-it (кнопка "Acknowledge license" -> "Access repository").
# НЕ коммить этот токен в git.
import os

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = 'hf_ВСТАВЬ_СВОЙ_ТОКЕН_СЮДА'

token = os.environ['HF_TOKEN'].strip()

# Диагностика без раскрытия токена в выводе.
masked = token[:3] + '*' * max(len(token) - 6, 0) + token[-3:] if len(token) > 6 else '(слишком короткий)'
print('длина:', len(token))
print('маска:', masked)

try:
    token.encode('ascii')
except UnicodeEncodeError as e:
    raise ValueError(
        f'Токен содержит не-ASCII символы ({e}). Плейсхолдер hf_ВСТАВЬ_СВОЙ_ТОКЕН_СЮДА заменён не целиком — '
        'выдели ВЕСЬ текст между кавычками и вставь токен вместо него.'
    )

os.environ['HF_TOKEN'] = token

from huggingface_hub import login, whoami
login(token=token)
me = whoami(token=token)
print('HF login OK, user:', me.get('name'))

In [ ]:
# --- Кэш в /workspace (переживает рестарт пода, / — всего 5GB) ---
os.environ['HF_HOME'] = '/workspace/hf_cache'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.makedirs('/workspace/hf_cache', exist_ok=True)
os.makedirs('/workspace/test_docs', exist_ok=True)
print('HF_HOME =', os.environ['HF_HOME'])

In [ ]:
# --- Проверка места на /workspace ---
# К этому моменту кэш уже мог накопить Qolda (~9GB) + Sherkala (~16GB) + KazLLM (~16GB) +
# Qwen3-VL-8B (~16GB) = ~57GB. Gemma 3 12B в bf16 добавит ещё ~24GB — если места не хватит,
# именно отсюда берётся "Disk quota exceeded" при скачивании.
!df -h /workspace
print()
!du -sh /workspace/hf_cache/models--* 2>/dev/null | sort -rh

In [ ]:
# --- Опционально: освободить место, удалив кэш уже проверенных моделей ---
# Выполнить ТОЛЬКО если df выше показал нехватку места и данные по этим моделям уже
# записаны в findings.md — сами веса на диске больше не нужны, HF перекачает при необходимости.
# Раскомментировать нужные строки:

# !rm -rf /workspace/hf_cache/models--issai--Qolda
# !rm -rf /workspace/hf_cache/models--inceptionai--Llama-3.1-Sherkala-8B-Chat
# !rm -rf /workspace/hf_cache/models--issai--LLama-3.1-KazLLM-1.0-8B
# !rm -rf /workspace/hf_cache/models--Qwen--Qwen3-VL-8B-Instruct

!df -h /workspace

In [ ]:
# --- Зависимости ---
# Gemma 3 поддерживается с transformers>=4.50.0. Берём ту же 4.55.0, что для Sherkala/KazLLM/Qolda —
# она новее минимума и совместима, для честности сравнения на одинаковом стеке.
%pip install -q "transformers==4.55.0" accelerate pillow hf_transfer

In [ ]:
import transformers
print('transformers version:', transformers.__version__)

## Загрузка модели

`Gemma3ForConditionalGeneration` + `AutoProcessor` — тот же паттерн, что у Qwen3-VL, без ручного тайлинга.

In [ ]:
import time
from transformers import Gemma3ForConditionalGeneration, AutoProcessor

MODEL_ID = 'google/gemma-3-12b-it'

t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Gemma3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
).eval().cuda()
print(f'Загружено за {time.time()-t0:.1f}s')
print('VRAM занято, GB:', round(torch.cuda.memory_allocated() / 2**30, 2))

## Тест 1 — языки (kk/ru/en)

Те же промпты, что во всех предыдущих ноутбуках — для прямого сравнения бок о бок по пяти моделям.

In [ ]:
GEN_KW = dict(max_new_tokens=512, do_sample=False)

def ask_text(question):
    messages = [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors='pt',
    )
    inputs = {k: (v.to(model.device) if hasattr(v, 'to') else v) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, **GEN_KW)
    new_tokens = out[0][inputs['input_ids'].shape[-1]:]
    return processor.decode(new_tokens, skip_special_tokens=True).strip()

prompts = {
    'kk': 'Қазақстанның бас қаласы қандай және неге астана болып таңдалды? Қысқаша жауап бер.',
    'ru': 'Объясни своими словами, чем отличается ИП от ТОО в Казахстане. Кратко.',
    'en': 'In two sentences, explain what a vector database is used for in RAG systems.',
}

for lang, q in prompts.items():
    t0 = time.time()
    answer = ask_text(q)
    dt = time.time() - t0
    print(f'=== {lang} ({dt:.1f}s) ===')
    print('Q:', q)
    print('A:', answer)
    print()

**На что смотреть:** если ответ на `kk` связный и грамматически чист — заявление про "140+ языков" подтверждается
на практике для казахского. Если модель путает язык, отвечает на русском/английском вместо казахского, или текст
выглядит как машинный перевод — это тот самый риск, о котором предупреждал исходный план ("kk не подтверждён\
официально").

## Тест 2 — зрение / OCR документа

Тот же тестовый файл, что в ноутбуках Qolda и Qwen3-VL (`/workspace/test_docs/sample.png`).

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

SAMPLE_PATH = Path('/workspace/test_docs/sample.png')

if not SAMPLE_PATH.exists():
    print('Реального скана нет — генерирую синтетический тестовый документ (тот же, что у Qolda/Qwen3-VL).')
    img = Image.new('RGB', (1000, 500), 'white')
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 28)
    except Exception:
        font = ImageFont.load_default()
    lines = [
        'АНЫҚТАМА / СПРАВКА',
        'Тегі: Ахметов А.Б.',
        'Мекенжайы: Алматы қаласы, Абай көшесі 10',
        'Дата выдачи: 06.08.2026',
        'Номер документа: 45231-KZ',
    ]
    for i, line in enumerate(lines):
        draw.text((40, 40 + i * 60), line, fill='black', font=font)
    img.save(SAMPLE_PATH)
    print('Синтетический документ сохранён:', SAMPLE_PATH)
else:
    print('Использую существующий файл:', SAMPLE_PATH)

img = Image.open(SAMPLE_PATH)
print('Размер:', img.size)
img

In [ ]:
def ask_with_image(image_path, question):
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': Image.open(image_path)},
            {'type': 'text', 'text': question},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors='pt',
    )
    inputs = {k: (v.to(model.device) if hasattr(v, 'to') else v) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, **GEN_KW)
    new_tokens = out[0][inputs['input_ids'].shape[-1]:]
    return processor.decode(new_tokens, skip_special_tokens=True).strip()

t0 = time.time()
answer = ask_with_image(
    SAMPLE_PATH,
    'Прочитай весь текст на изображении дословно, ничего не придумывай. Если что-то нечитаемо — так и скажи.',
)
dt = time.time() - t0
print(f'--- Ответ ({dt:.1f}s) ---')
print(answer)

**Сравнить с Qolda (`01_qolda_check.ipynb`) и Qwen3-VL (`04_qwen3vl_check.ipynb`)** на одном и том же документе —
третья точка для вопроса "насколько сильно специализация Qolda под kk окупается против общих VL-моделей".

## Замер VRAM

In [ ]:
print('VRAM allocated, GB:', round(torch.cuda.memory_allocated() / 2**30, 2))
print('VRAM reserved,  GB:', round(torch.cuda.memory_reserved() / 2**30, 2))
print('VRAM peak,      GB:', round(torch.cuda.max_memory_allocated() / 2**30, 2))

In [ ]:
# Освободить память, если на этом поде дальше запускается что-то ещё.
del model
torch.cuda.empty_cache()
print('VRAM после очистки, GB:', round(torch.cuda.memory_allocated() / 2**30, 2))

## Дальше по плану

Пятая и, вероятно, последняя точка сравнения в стенде. Свести все пять моделей (Qolda, Sherkala, KazLLM,
Qwen3-VL, Gemma 3 12B) в единую таблицу `docs/findings.md`: языки kk/ru/en, зрение (где применимо), VRAM,
latency, лицензия, tool calling. На этом набранных данных должно хватить для решения, какая модель идёт
в каркас `app/`.